In [1]:
import re
import pandas as pd
import numpy as np
import warnings
import os
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
from sql_db_func import  MySQLConnetFunc
dbhandler = MySQLConnetFunc('l6a01_project')

In [12]:
tbns = dbhandler.list_tables()

day_df = dbhandler.get_table( 'aoi100_capa_summary')
hourly_df = dbhandler.get_table('aoi100_capa_hourly_rawdata')
sdf = dbhandler.get_table('aoi_summary_aoi100')
pi_df = dbhandler.get_table('aoi100_pidensity_202511_pi500')
tbns[:3]

2025-11-26 13:08:09,984 - INFO - [list_tables] 成功取得資料表名稱，共 93 張表。


['aoi100_capa_hourly_rawdata',
 'aoi100_capa_summary',
 'aoi100_glassdata_202511']

In [8]:
tbns

['aoi100_capa_hourly_rawdata',
 'aoi100_capa_summary',
 'aoi100_glassdata_202511',
 'aoi100_pidensity_202509_pi000',
 'aoi100_pidensity_202509_pi100',
 'aoi100_pidensity_202509_pi200',
 'aoi100_pidensity_202509_pi300',
 'aoi100_pidensity_202509_pi400',
 'aoi100_pidensity_202509_pi500',
 'aoi100_pidensity_202509_pi600',
 'aoi100_pidensity_202509_pi700',
 'aoi100_pidensity_202510_pi000',
 'aoi100_pidensity_202510_pi100',
 'aoi100_pidensity_202510_pi200',
 'aoi100_pidensity_202510_pi300',
 'aoi100_pidensity_202510_pi400',
 'aoi100_pidensity_202510_pi500',
 'aoi100_pidensity_202510_pi600',
 'aoi100_pidensity_202510_pi700',
 'aoi100_pidensity_202511_pi000',
 'aoi100_pidensity_202511_pi100',
 'aoi100_pidensity_202511_pi200',
 'aoi100_pidensity_202511_pi300',
 'aoi100_pidensity_202511_pi400',
 'aoi100_pidensity_202511_pi500',
 'aoi100_pidensity_202511_pi600',
 'aoi100_pidensity_202511_pi700',
 'aoi100_rawdata_202508',
 'aoi100_rawdata_202509',
 'aoi100_rawdata_202510',
 'aoi100_rawdata_202511

In [13]:
pi_df .iloc[100,:].to_dict()

{'scan_time': Timestamp('2025-11-04 02:36:22'),
 'line_id': 'CAPIC500',
 'model': 'M270DAN8S',
 'glass_type': 'CF',
 'recipe_id': '205',
 'glass_id': 'AR0DRA0BFA',
 'pic_name': 'CaptureImage/CAPIT203_AR0DRA0BFA_2_12_2511040236.JPG',
 'x': '1472140',
 'y': '241247',
 'predict_code': '!',
 'judge_code': '!',
 'mark': '',
 'hour': '',
 'dayhour': '',
 'day': Timestamp('2025-11-04 02:36:22'),
 'year': '',
 'month': '',
 'season': '',
 'week': '',
 'yearmonth': '',
 'defect_count': '18',
 'defect_size': '24',
 'open_status': '',
 'ai_code_1': 'NPI_TFT',
 'ai_code_2': '2025-11-03 18:58:49',
 'ai_code_3': '',
 'ai_code_4': '',
 'ai_code_5': '',
 'gray_name': 'CCDImage\\IP01\\0022',
 'ip_num': '',
 'first_code': '',
 'chip_name': 'AR0DRA0BFAA',
 'defect_seq': '2',
 'pi_time': '2025-11-03 19:50:09',
 'pi_hour': '2025-11-03 19',
 'pic_path': 'http://10.97.139.98:1454/CAAOI202/CA007001/JF5AATC8C/PCS1/20251104051010/',
 'recipe_comment': ''}

In [ ]:
# ===== 連線設定）=====
user_name    = "L6AINT_AP"
passwd       = "L6AINT$AP"
port         = "1549"
host         = "TCPPA104"
service_name = "L6AHSHA"
DATABASE_URL = f"oracle+cx_oracle://{user_name}:{passwd}@{host}:{port}/?service_name={service_name}"

engine = create_engine(
    DATABASE_URL,
    connect_args={"encoding": "UTF-8", "nencoding": "UTF-8", "events": True}
)

In [5]:
class OracleConfig:
    def __init__(self):
        user_name    = "L6AINT_AP"
        passwd       = "L6AINT$AP"
        port         = "1549"
        host         = "TCPPA104"
        service_name = "L6AHSHA"
        # 建立連線字串
        self.DATABASE_URL = f"oracle+cx_oracle://{user_name}:{passwd}@{host}:{port}/?service_name={service_name}"
        self.create_engine_func()

        #======Oracle 資料表欄位========
        #"H_AIDI_SECDEFECT"
        self.H_AIDI_SECDEFECT_COLS = ['create_timestamp', 'lm_time', 'lm_user', 'fab_code', 'process_stage',
       'sheet_id_chip_id', 'chip_id', 'test_time', 'measure_category',
       'chip_seq_no', 'defect_seq_no', 'defect_size', 'defect_area',
       'defect_shape', 'defect_value', 'defect_type', 'signal_no', 'gate_no',
       'pox_x1', 'pox_y1', 'pox_x2', 'pox_y2', 'image_file_path',
       'image_file_name', 'img_file_url_path', 'image_start_seq', 'image_qty',
       'image_flag', 'image_load_count', 'image_load_status',
       'aoi_def_code_cate', 'aoi_def_code', 'retype_def_code',
       'retype_def_code_desc', 'retype_user', 'retype_time',
       'current_def_code', 'current_def_code_desc', 'repair_code',
       'repair_flag', 'adc_accept_flag', 'adc_accept_time', 'adc_answers',
       'adc_def_code', 'adc_def_code_desc', 'adc_is_bypass', 'adc_judge_flag',
       'adc_judge_return_code', 'adc_judge_time', 'adc_repair_flag',
       'feature_regions', 'feature_region_count', 'adc_score']
        
        

        
        self.oracle_defetc_cols = ['create_timestamp', 'sheet_id_chip_id' ,'image_file_path',
            'image_file_name', 'img_file_url_path']
        


        #"H_AIDI_SECSHEETCHIP"

        self.H_AIDI_SECSHEETCHIP_COLS = ['create_timestamp', 'lm_time', 'lm_user', 'fab_code', 'process_stage',
       'sheet_id_chip_id', 'test_time', 'measure_category', 'product_code',
       'model_no', 'abbr_no', 'abbr_cat', 'recipe_id', 'cassette_id', 'op_id',
       'line_id', 'eqp_id', 'eqp_start_time', 'eqp_end_time',
       'process_line_id', 'process_eqp_id', 'judge', 'total_defect_image_qty',
       'total_defect_qty', 'image_load_qty', 'total_chip_qty', 'insp_chip_qty',
       'defect_size_o_qty', 'defect_size_l_qty', 'defect_size_m_qty',
       'defect_size_s_qty', 'load_complete_flag', 'load_complete_time',
       'pixel_length', 'complete_time', 'complete_userid', 'retype_flag',
       'retype_time', 'retype_user', 'adc_model_id', 'adc_scope',
       'adc_topn_result', 'adc_accept_flag', 'adc_accept_image_qty',
       'adc_accept_time', 'adc_judge_flag', 'adc_judge_time',
       'adc_judge_image_qty']
        
        self.oracle_chip_cols = ['create_timestamp','sheet_id_chip_id','product_code',
            'model_no', 'abbr_no', 'recipe_id', 'cassette_id', 'op_id',
            'line_id', 'eqp_id', 'eqp_start_time', 'eqp_end_time','total_defect_image_qty',
            'total_defect_qty', 'image_load_qty','defect_size_o_qty', 'defect_size_l_qty', 'defect_size_m_qty',
            'defect_size_s_qty', 'load_complete_flag']

        self.oracle_table_columns_dict = {"H_AIDI_SECDEFECT": self.oracle_defetc_cols,
                                    "H_AIDI_SECSHEETCHIP": self.oracle_chip_cols}


    def create_engine_func(self):
        self.engine = create_engine(
            DATABASE_URL,
            connect_args={
                'encoding': 'UTF-8',
                'nencoding': 'UTF-8',
                'events': True
            }
        )


cfg = OracleConfig()

In [ ]:
def get_timedelta_data(config, d, table_name):
    table_cols= config.oracle_table_columns_dict[table_name]
    now = datetime.now()
    print(f"連線Oracle 時間: {now}")
    delta_ago = now - timedelta(days=d)

    # 用 f-string 插入表名（已先驗證）
    query = text(f"""
        SELECT *
        FROM CELAIDI.{table_name.upper()}
        WHERE CREATE_TIMESTAMP >= TO_DATE(:start_time, 'YYYY-MM-DD HH24:MI:SS')
    """)

    df = pd.read_sql(query, 
                     con=config.engine, 
                     params={"start_time": delta_ago.strftime('%Y-%m-%d %H:%M:%S')})
    print(f'{table_name}, cols:{len(df.columns)}, rows: {len(df)}')
    # 把 None 換成 NaN，刪掉整列為 NaN 的資料
    df = df.replace({None: np.nan}).dropna(how='all')
    #df = df[table_cols]
    """
    df['create_timestamp'] = pd.to_datetime(df['create_timestamp'], errors = 'coerce')
    df['create_timestamp'] = df['create_timestamp'].dt.floor('S')
    #o_defect_df['datetime_str'] = pd.to_datetime(o_defect_df['create_timestamp']).dt.strftime('%Y%m%d%H%M%S')
    df = df.sort_values(by = 'create_timestamp', ascending=False)
    #df[(df['eqp_id'] == 'CAAOI202') & (df['op_id'] == 'PCS1')] ###TEST###
    """
    """
    for cl in df.columns:
        if not df[df[cl]=="CA038501"].empty:
            print(cl)
            print(df[df[cl]=="CA038501"])
    
    
    df['create_date'] = df['create_timestamp'].dt.strftime('%Y%m%d%H%M%S')
    #df = df.iloc[13:,:].reset_index(drop = True)
    """
    return df #.to_dict(orient='records')

sec_df = get_timedelta_data(cfg, 3, "H_AIDI_SECDEFECT")
sec_chip_df = get_timedelta_data(cfg, 3, "H_AIDI_SECSHEETCHIP")
#print(o_DEFECT_df.columns)


連線Oracle 時間: 2025-11-26 10:35:23.676804
H_AIDI_SECDEFECT Index(['create_timestamp', 'lm_time', 'lm_user', 'fab_code', 'process_stage',
       'sheet_id_chip_id', 'chip_id', 'test_time', 'measure_category',
       'chip_seq_no', 'defect_seq_no', 'defect_size', 'defect_area',
       'defect_shape', 'defect_value', 'defect_type', 'signal_no', 'gate_no',
       'pox_x1', 'pox_y1', 'pox_x2', 'pox_y2', 'image_file_path',
       'image_file_name', 'img_file_url_path', 'image_start_seq', 'image_qty',
       'image_flag', 'image_load_count', 'image_load_status',
       'aoi_def_code_cate', 'aoi_def_code', 'retype_def_code',
       'retype_def_code_desc', 'retype_user', 'retype_time',
       'current_def_code', 'current_def_code_desc', 'repair_code',
       'repair_flag', 'adc_accept_flag', 'adc_accept_time', 'adc_answers',
       'adc_def_code', 'adc_def_code_desc', 'adc_is_bypass', 'adc_judge_flag',
       'adc_judge_return_code', 'adc_judge_time', 'adc_repair_flag',
       'feature_regions', 

C:\Users\Ray\AppData\Local\Temp\ipykernel_25104\2342864048.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({None: np.nan}).dropna(how='all')


連線Oracle 時間: 2025-11-26 10:36:05.198782
H_AIDI_SECSHEETCHIP Index(['create_timestamp', 'lm_time', 'lm_user', 'fab_code', 'process_stage',
       'sheet_id_chip_id', 'test_time', 'measure_category', 'product_code',
       'model_no', 'abbr_no', 'abbr_cat', 'recipe_id', 'cassette_id', 'op_id',
       'line_id', 'eqp_id', 'eqp_start_time', 'eqp_end_time',
       'process_line_id', 'process_eqp_id', 'judge', 'total_defect_image_qty',
       'total_defect_qty', 'image_load_qty', 'total_chip_qty', 'insp_chip_qty',
       'defect_size_o_qty', 'defect_size_l_qty', 'defect_size_m_qty',
       'defect_size_s_qty', 'load_complete_flag', 'load_complete_time',
       'pixel_length', 'complete_time', 'complete_userid', 'retype_flag',
       'retype_time', 'retype_user', 'adc_model_id', 'adc_scope',
       'adc_topn_result', 'adc_accept_flag', 'adc_accept_image_qty',
       'adc_accept_time', 'adc_judge_flag', 'adc_judge_time',
       'adc_judge_image_qty'],
      dtype='object')


In [27]:
print(len(sec_df))
def clean_process(df):
    df['create_timestamp'] = pd.to_datetime(df['create_timestamp'], errors = 'coerce')
    df['create_timestamp'] = df['create_timestamp'].dt.floor('S')
    #o_defect_df['datetime_str'] = pd.to_datetime(o_defect_df['create_timestamp']).dt.strftime('%Y%m%d%H%M%S')
    df = df.sort_values(by = 'create_timestamp', ascending=False)
    return df
#sec_df.head()
sec_df2 = clean_process(sec_chip_df)
sec_df2.head()

322076


C:\Users\Ray\AppData\Local\Temp\ipykernel_25104\2153442792.py:4: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['create_timestamp'] = df['create_timestamp'].dt.floor('S')


,create_timestamp,lm_time,lm_user,fab_code,process_stage,sheet_id_chip_id,test_time,measure_category,product_code,model_no,...,retype_user,adc_model_id,adc_scope,adc_topn_result,adc_accept_flag,adc_accept_image_qty,adc_accept_time,adc_judge_flag,adc_judge_time,adc_judge_image_qty
172,2025-11-26 10:25:13,2025-11-26 10:27:22.344,adc,101600,FEOL,AR0QF500FX,2025-11-26 10:20:53,PIT,M270QAN06,M270QAN6J_VB,...,adc,S_L6A_DT_GD_PI_AOI,"{""model_no"":""M270QAN6J*"",""op_id"":""ALL""}",NaN,Y,246,2025-11-26 10:25:34.514,Y,2025-11-26 10:27:22.344,NaN
171,2025-11-26 10:23:35,2025-11-26 10:26:29.484,adc,101600,CELL,VP5GAHK1Q,2025-11-26 10:17:50,PIT,M340QAR1B_VM,M340QAR1B,...,adc,S_L6A_DT_GD_PI_AOI,"{""model_no"":""M340QAR1B*"",""op_id"":""ALL""}",NaN,Y,326,2025-11-26 10:23:34.514,Y,2025-11-26 10:26:29.484,NaN
170,2025-11-26 10:21:09,2025-11-26 10:24:30.958,adc,101600,FEOL,AR0QF500M1,2025-11-26 10:15:38,PIT,M270QAN06,M270QAN6J_VB,...,adc,S_L6A_DT_GD_PI_AOI,"{""model_no"":""M270QAN6J*"",""op_id"":""ALL""}",NaN,Y,336,2025-11-26 10:22:34.514,Y,2025-11-26 10:24:30.958,NaN
169,2025-11-26 10:19:35,2025-11-26 10:23:20.675,adc,101600,CELL,VP5GAHK1Q,2025-11-26 10:13:09,PIT,M340QAR1B_VM,M340QAR1B,...,adc,S_L6A_DT_GD_PI_AOI,"{""model_no"":""M340QAR1B*"",""op_id"":""ALL""}",NaN,Y,335,2025-11-26 10:20:34.514,Y,2025-11-26 10:23:20.675,NaN
168,2025-11-26 10:15:10,2025-11-26 10:16:34.041,adc,101600,FEOL,AR0QF5009E,2025-11-26 10:11:22,PIT,M270QAN06,M270QAN6J_VB,...,adc,S_L6A_DT_GD_PI_AOI,"{""model_no"":""M270QAN6J*"",""op_id"":""ALL""}",NaN,Y,210,2025-11-26 10:15:34.511,Y,2025-11-26 10:16:34.041,NaN


In [31]:
cols = ['create_timestamp', 'model_no', 'line_id','recipe_id','sheet_id_chip_id']
sec_df3 = sec_df2[cols]
sec_df3[sec_df3['sheet_id_chip_id'].str.contains('AR0QFA00EE')]

,create_timestamp,model_no,line_id,recipe_id,sheet_id_chip_id
123,2025-11-26 08:48:13,M270QAN6J_VB,CAPIT203,605,AR0QFA00EE


In [36]:
"""
http://10.97.139.98:1454/CAAOI202//YD5A7NA2A/PCS1/20250728021055/CaptureImage/RV1_1645179_1444405_32.JPG
http://10.97.139.98:1454/CAAOI202/LOTID/SHEETID/OPID/檢測時間/CaptureImage/照片檔名

"""
img_root = 'http://10.97.139.98:1454/CAAOI202/'
['sheet_id_chip_id','op_id']

lot_id = 'YD5A7NA2A'
lot_defect_df = o_defect_df[o_defect_df['sheet_id_chip_id'].str.contains(lot_id) ]#[cfg.oracle_defetc_cols] #& 
lot_df = o_runchip_df[o_runchip_df['sheet_id_chip_id'].str.contains(lot_id)]

In [41]:
lot_defect_df.iloc[:1,:].to_dict(orient = 'index')

{452673: {'create_timestamp': Timestamp('2025-07-28 02:23:55'),
  'lm_time': Timestamp('2025-07-28 02:23:54.256000'),
  'lm_user': nan,
  'fab_code': 101600,
  'process_stage': 'CELL',
  'sheet_id_chip_id': 'YD5A7NA2A',
  'chip_id': 'H',
  'test_time': Timestamp('2025-07-28 02:01:58'),
  'measure_category': 'PIT',
  'chip_seq_no': '020',
  'defect_seq_no': '1371',
  'defect_size': 'M',
  'defect_area': '40',
  'defect_shape': nan,
  'defect_value': nan,
  'defect_type': nan,
  'signal_no': 234.0,
  'gate_no': 85.0,
  'pox_x1': 260982,
  'pox_y1': 326355,
  'pox_x2': nan,
  'pox_y2': nan,
  'image_file_path': 'Image/CA038501/YD5A7NA2A/',
  'image_file_name': '04_01_0052.jpg',
  'img_file_url_path': 'PIT/2507/28/CAAOI202/YD5A7NA2A/0201/',
  'image_start_seq': nan,
  'image_qty': nan,
  'image_flag': 'Y',
  'image_load_count': 1.0,
  'image_load_status': 'N',
  'aoi_def_code_cate': nan,
  'aoi_def_code': '-',
  'retype_def_code': nan,
  'retype_def_code_desc': nan,
  'retype_user': nan,
 

In [38]:
lot_df.to_dict(orient = 'index')

{10707: {'create_timestamp': Timestamp('2025-07-28 02:13:48'),
  'lm_time': Timestamp('2025-07-28 15:52:25.577000'),
  'lm_user': '0503474',
  'fab_code': 101600,
  'process_stage': 'CELL',
  'sheet_id_chip_id': 'YD5A7NA2A',
  'test_time': Timestamp('2025-07-28 02:01:58'),
  'measure_category': 'PIT',
  'product_code': 'B160UAN4A_VB',
  'model_no': 'B160UAN4A',
  'abbr_no': nan,
  'abbr_cat': 'TFT',
  'recipe_id': '2251',
  'cassette_id': 'CA0385',
  'op_id': 'PCS1',
  'line_id': nan,
  'eqp_id': 'CAAOI202',
  'eqp_start_time': Timestamp('2025-07-28 02:01:58'),
  'eqp_end_time': Timestamp('2025-07-28 02:10:48'),
  'process_line_id': nan,
  'process_eqp_id': nan,
  'judge': nan,
  'total_defect_image_qty': 1507,
  'total_defect_qty': 1507,
  'image_load_qty': 1000,
  'total_chip_qty': nan,
  'insp_chip_qty': nan,
  'defect_size_o_qty': 0,
  'defect_size_l_qty': 673,
  'defect_size_m_qty': 492,
  'defect_size_s_qty': 269,
  'load_complete_flag': 'E',
  'load_complete_time': Timestamp('20